# CityBrain v7 — Enhanced Feature Engineering + 3-Model Ensemble


## CityBrain v7 — Enhanced Feature Engineering + 3-Model Ensemble


In [96]:
# Changes from v6:
#   1. Spatial lag features (7-NN inverse-distance weighted, train-only)
#   2. Neighbourhood target encoding (train-only, no leakage)
#   3. Road-MLP restored to 13d (was 6d) — add back repair, density, source, traffic + spatial
#   4. Tabular-MLP expanded to ~35d — add spatial lag + neigh encoding
#   5. RandomOverSampler for minority class balancing
#   6. Slimmer CrossAttention (d_model=32, was 64) — reduce overfitting
#   7. LightGBM as third ensemble member
#   8. 3-model ensemble weight search on VALIDATION set (not test)
#   9. Per-class threshold optimisation (differential_evolution)
#  10. Fixed: ensemble weights tuned on val, applied to test

##


In [97]:
# ============================================================
# 0. Setup
# ============================================================
import os, ast, warnings
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
warnings.filterwarnings('ignore')

# --- Colab ---
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/AI-FinalProject/data'

SNAP_RADIUS_M = 150
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ★ REBALANCED 3-CLASS LABEL MAP ★
LABEL_MAP = {
    'VERY GOOD': 0,                # Low
    'GOOD': 1, 'FAIR': 1,          # Medium
    'POOR': 2, 'VERY POOR': 2,     # High
}
NUM_CLASSES = 3
CLASS_NAMES = ['Low', 'Medium', 'High']

def safe_load(filename, **kwargs):
    path = os.path.join(DATA_DIR, filename) if not os.path.isabs(filename) else filename
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        first = f.readline()
    sep = ';' if first.count(';') > first.count(',') else ','
    return pd.read_csv(path, sep=sep, on_bad_lines='skip', **kwargs)

print(f'Label mapping: {LABEL_MAP}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Label mapping: {'VERY GOOD': 0, 'GOOD': 1, 'FAIR': 1, 'POOR': 2, 'VERY POOR': 2}
Classes (3): ['Low', 'Medium', 'High']


## 1. Load & Merge Data


In [98]:
enriched_path = '/content/drive/MyDrive/AI-FinalProject/data/pavement_enriched.csv'
if os.path.exists(enriched_path):
    df = pd.read_csv(enriched_path)
    print(f'Loaded pavement_enriched.csv: {len(df):,} rows')
else:
    df_local = safe_load('pavement_condition.csv'); df_local['road_type'] = 'local'
    df_major = safe_load('pavement_condition_major_road_network.csv'); df_major['road_type'] = 'major'
    df = pd.concat([df_local, df_major], ignore_index=True)
    coords = df['geo_point_2d'].str.split(',', expand=True).astype(float)
    df['lat'], df['lon'] = coords[0], coords[1]

df = df[df['PCI Rating'].isin(LABEL_MAP)].copy()
df['risk_label'] = df['PCI Rating'].map(LABEL_MAP)
df['source'] = (df['road_type'] == 'major').astype(int) if 'road_type' in df.columns else 0
print(f'Total samples: {len(df):,} (ALL kept)')
print(f'\nLabel distribution:')
for v, name in enumerate(CLASS_NAMES):
    cnt = (df['risk_label']==v).sum()
    print(f'  {v} ({name}): {cnt:,} ({cnt/len(df)*100:.1f}%)')

pave_coords = np.column_stack([df['lat'].values, df['lon'].values])
LAT_M, LON_M = 111_000, 73_000

# --- streetuse ---
df_st = pd.read_csv('/content/drive/MyDrive/AI-FinalProject/data/public_streets.csv',
                     quotechar='"', on_bad_lines='skip')
st_geo = df_st['geo_point_2d'].dropna().apply(ast.literal_eval)
st_coords = np.array([[d['lat'], d['lon']] for d in st_geo])
_, idx = cKDTree(st_coords).query(pave_coords)
df['streetuse'] = df_st.loc[st_geo.index, 'streetuse'].values[idx]
STREETUSE_WEIGHT = {'Residential': 0.3, 'Collector': 0.6, 'Arterial': 1.0}
df['streetuse_weight'] = df['streetuse'].map(STREETUSE_WEIGHT).fillna(0.5)
print('streetuse:', df['streetuse'].value_counts().to_dict())

# --- ROW width ---
df_row = safe_load('right_of_way_widths.csv')
geo_col = [c for c in df_row.columns if 'geo_point' in c.lower()][0]
width_col = [c for c in df_row.columns if 'width' in c.lower()][0]
row_geo = df_row[geo_col].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
row_coords = np.array([[d['lat'], d['lon']] for d in row_geo if isinstance(d, dict)])
row_widths = pd.to_numeric(df_row.loc[row_geo.index[:len(row_coords)], width_col], errors='coerce').fillna(0).values
_, idx = cKDTree(row_coords).query(pave_coords)
df['ROW_width'] = row_widths[idx]

# --- repair_count ---
df['repair_count'] = 0
repair_path = '/content/drive/MyDrive/AI-FinalProject/data/city_project_package_street.csv'
if os.path.exists(repair_path):
    df_repair = safe_load('city_project_package_street.csv')
    geo_cols = [c for c in df_repair.columns if 'geo_point' in c.lower()]
    if geo_cols:
        try:
            rg = df_repair[geo_cols[0]].dropna()
            parsed = rg.apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
            rc = np.array([[d['lat'], d['lon']] for d in parsed if isinstance(d, dict)])
            counts = cKDTree(rc * [LAT_M, LON_M]).query_ball_point(pave_coords * [LAT_M, LON_M], r=100)
            df['repair_count'] = [len(c) for c in counts]
        except: pass

# --- segment_density ---
tree_all = cKDTree(pave_coords)
neighbours = tree_all.query_ball_point(pave_coords, r=0.003)
df['segment_density'] = [len(n) - 1 for n in neighbours]

# --- elevation & slope ---
if 'elevation_m' not in df.columns:
    df['elevation_m'] = 0.0; df['slope_pct'] = 0.0
df['elevation_m'] = df['elevation_m'].fillna(df['elevation_m'].median())
df['slope_pct'] = df['slope_pct'].fillna(df['slope_pct'].median())

# --- neighbourhood ---
df['neighbourhood'] = df.get('neighbourhood', pd.Series('Unknown', index=df.index)).fillna('Unknown')

# --- traffic counter distance ---
df_traffic = safe_load('directional_traffic_count_locations.csv')
traf_geo_col = [c for c in df_traffic.columns if 'geo_point' in c.lower()][0]
traf_geo = df_traffic[traf_geo_col].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
traf_coords = np.array([[d['lat'], d['lon']] for d in traf_geo if isinstance(d, dict)])
traf_dists, _ = cKDTree(traf_coords * [LAT_M, LON_M]).query(pave_coords * [LAT_M, LON_M])
df['dist_to_traffic_count'] = traf_dists

print(f'Static features ready.')

Loaded pavement_enriched.csv: 13,764 rows
Total samples: 13,764 (ALL kept)

Label distribution:
  0 (Low): 3,247 (23.6%)
  1 (Medium): 5,894 (42.8%)
  2 (High): 4,623 (33.6%)
streetuse: {'Arterial': 5877, 'Residential': 5180, 'Collector': 2707}
Static features ready.


## 2. Train / Val / Test Split


In [99]:
y = df['risk_label'].values
indices = np.arange(len(df))
idx_train, idx_temp = train_test_split(indices, test_size=0.30, random_state=42, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42, stratify=y[idx_temp])
print(f'Train: {len(idx_train):,}  Val: {len(idx_val):,}  Test: {len(idx_test):,}')
for split_name, idxs in [('Train', idx_train), ('Val', idx_val), ('Test', idx_test)]:
    counts = np.bincount(y[idxs], minlength=NUM_CLASSES)
    parts = ', '.join(f'{CLASS_NAMES[c]}={counts[c]:,}' for c in range(NUM_CLASSES))
    print(f'  {split_name}: {parts}')

Train: 9,634  Val: 2,065  Test: 2,065
  Train: Low=2,273, Medium=4,125, High=3,236
  Val: Low=487, Medium=884, High=694
  Test: Low=487, Medium=885, High=693


## 3. ★ NEW: Spatial Lag Features (train-only, no leakage) ★


In [100]:
print('\n--- Computing Spatial Lag Features ---')
train_coords_m = pave_coords[idx_train] * [LAT_M, LON_M]
all_coords_m   = pave_coords * [LAT_M, LON_M]
train_tree_sl  = cKDTree(train_coords_m)
K_SL = 7
# ★ Query K+1 neighbours, then drop self-matches to prevent leakage ★
dists_k, nn_k  = train_tree_sl.query(all_coords_m, k=K_SL + 1)
self_match     = dists_k[:, 0] < 0.01       # training samples match themselves (dist≈0)
dists_k = np.where(self_match[:, None], dists_k[:, 1:K_SL+1], dists_k[:, :K_SL])
nn_k    = np.where(self_match[:, None], nn_k[:, 1:K_SL+1],    nn_k[:, :K_SL])
print(f'  Self-matches removed: {self_match.sum()} / {len(self_match)}')

train_labels   = y[idx_train]
sl_weights     = 1.0 / (dists_k + 1e-6)
sl_weights_n   = sl_weights / sl_weights.sum(axis=1, keepdims=True)
neighbour_labs = train_labels[nn_k]

df['spatial_lag_risk'] = (sl_weights_n * neighbour_labs).sum(axis=1)
df['spatial_lag_high'] = (sl_weights_n * (neighbour_labs == 2)).sum(axis=1)
df['spatial_lag_med']  = (sl_weights_n * (neighbour_labs == 1)).sum(axis=1)
print(f'  spatial_lag_risk: mean={df["spatial_lag_risk"].mean():.3f}, std={df["spatial_lag_risk"].std():.3f}')
print(f'  spatial_lag_high: mean={df["spatial_lag_high"].mean():.3f}')
print(f'  spatial_lag_med:  mean={df["spatial_lag_med"].mean():.3f}')


--- Computing Spatial Lag Features ---
  Self-matches removed: 9634 / 13764
  spatial_lag_risk: mean=1.099, std=0.453
  spatial_lag_high: mean=0.336
  spatial_lag_med:  mean=0.427


## 4. ★ NEW: Neighbourhood Target Encoding (train-only) ★


In [101]:
print('\n--- Computing Neighbourhood Target Encoding ---')
train_df = df.iloc[idx_train]
for cls, name in [(0, 'neigh_low_pct'), (1, 'neigh_med_pct'), (2, 'neigh_high_pct')]:
    d = train_df.groupby('neighbourhood')['risk_label'].apply(
        lambda x: (x == cls).mean()).to_dict()
    global_rate = (train_df['risk_label'] == cls).mean()
    df[name] = df['neighbourhood'].map(d).fillna(global_rate)
print(f'  neigh_high_pct range: [{df["neigh_high_pct"].min():.3f}, {df["neigh_high_pct"].max():.3f}]')
print(f'  neigh_med_pct  range: [{df["neigh_med_pct"].min():.3f}, {df["neigh_med_pct"].max():.3f}]')
print(f'  neigh_low_pct  range: [{df["neigh_low_pct"].min():.3f}, {df["neigh_low_pct"].max():.3f}]')


--- Computing Neighbourhood Target Encoding ---
  neigh_high_pct range: [0.214, 0.500]
  neigh_med_pct  range: [0.240, 0.531]
  neigh_low_pct  range: [0.129, 0.500]


## 5. Weather & 311 Temporal Tensor


In [102]:
df_w = safe_load('weather_vancouver.csv')
date_col = [c for c in df_w.columns if 'date' in c.lower() or 'time' in c.lower()][0]
df_w['date'] = pd.to_datetime(df_w[date_col], errors='coerce')
df_w = df_w.dropna(subset=['date'])
col_map = {}
for c in df_w.columns:
    cl = c.lower().strip()
    if 'max temp' in cl and 'flag' not in cl: col_map[c] = 'max_temp'
    elif 'min temp' in cl and 'flag' not in cl: col_map[c] = 'min_temp'
    elif 'total precip' in cl and 'flag' not in cl: col_map[c] = 'total_precip'
    elif 'total snow' in cl and 'flag' not in cl: col_map[c] = 'total_snow'
df_w = df_w.rename(columns=col_map)
for c in ['max_temp','min_temp','total_precip','total_snow']:
    if c in df_w.columns: df_w[c] = pd.to_numeric(df_w[c], errors='coerce').fillna(0)
df_w['freeze_thaw'] = ((df_w['max_temp'] > 0) & (df_w['min_temp'] < 0)).astype(int)
df_w['extreme'] = (df_w['total_precip'] > df_w['total_precip'].quantile(0.95)).astype(int)
df_w['year'], df_w['month'] = df_w['date'].dt.year, df_w['date'].dt.month
monthly_wx = df_w.groupby(['year','month']).agg(
    avg_max_temp=('max_temp','mean'), avg_min_temp=('min_temp','mean'),
    total_precip_mm=('total_precip','sum'), total_snow_cm=('total_snow','sum'),
    freeze_thaw_days=('freeze_thaw','sum'), extreme_days=('extreme','sum'),
).reset_index()
print(f'Weather: {len(df_w):,} days')

# --- 311 (pavement-related only) ---
print('\nLoading 311...')
df_311 = safe_load('311_service_requests_2009_2021.csv', low_memory=False)
id_cols = [c for c in df_311.columns if 'case' in c.lower() or 'id' in c.lower()]
if id_cols: df_311 = df_311.drop_duplicates(subset=id_cols[0])
PAVEMENT_TYPES = [
    'Pothole Case', 'Street Repair Case', 'Sidewalk Repair Case',
    'Pavement Marking Maintenance Case', 'Pavement Markings Case',
    'Street Surface Water Flooding Case', 'Street Construction Concern Case',
    'Street Cleaning and Debris Pick Up Case', 'Curb Ramp Request Case',
]
type_col = 'Service request type'
df_311 = df_311[df_311[type_col].isin(PAVEMENT_TYPES)]
print(f'  Pavement-related: {len(df_311):,}')

date_cols = [c for c in df_311.columns if 'open' in c.lower() or 'timestamp' in c.lower() or 'created' in c.lower()]
if not date_cols: date_cols = [c for c in df_311.columns if 'date' in c.lower()]
df_311['date'] = pd.to_datetime(df_311[date_cols[0]], errors='coerce', utc=True)
lat_col = [c for c in df_311.columns if c.lower() in ['latitude','lat']]
lon_col = [c for c in df_311.columns if c.lower() in ['longitude','lon']]
if lat_col and lon_col:
    df_311['c_lat'] = pd.to_numeric(df_311[lat_col[0]], errors='coerce')
    df_311['c_lon'] = pd.to_numeric(df_311[lon_col[0]], errors='coerce')
else:
    geo_col = [c for c in df_311.columns if 'geo_local' in c.lower() or 'geo_point' in c.lower()]
    if geo_col:
        raw = df_311[geo_col[0]].dropna()
        if len(raw) > 0 and '{' in str(raw.iloc[0]):
            parsed = raw.apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else s)
            df_311.loc[raw.index, 'c_lat'] = [d.get('lat',np.nan) if isinstance(d,dict) else np.nan for d in parsed]
            df_311.loc[raw.index, 'c_lon'] = [d.get('lon',np.nan) if isinstance(d,dict) else np.nan for d in parsed]

df_311 = df_311.dropna(subset=['c_lat','c_lon','date'])
df_311['year'], df_311['month'] = df_311['date'].dt.year, df_311['date'].dt.month
survey_years = df['Year'].unique() if 'Year' in df.columns else [2020, 2021]
df_311 = df_311[df_311['year'].isin(survey_years)]
print(f'  Survey-year 311: {len(df_311):,}')

# Snap 311 to nearest pavement segment
c311_coords = df_311[['c_lat','c_lon']].values
pave_tree_m = cKDTree(pave_coords * [LAT_M, LON_M])
dists, snap_idx = pave_tree_m.query(c311_coords * [LAT_M, LON_M])
mask = dists < SNAP_RADIUS_M
df_311 = df_311[mask].copy()
df_311['seg_idx'] = snap_idx[mask]
df_311['is_pothole'] = df_311[type_col].str.contains('Pothole', case=False).astype(int)
print(f'  Snapped within {SNAP_RADIUS_M}m: {len(df_311):,}')

complaint_monthly = df_311.groupby(['seg_idx','year','month']).agg(
    cnt=('seg_idx','size'), pothole_cnt=('is_pothole','sum')
).reset_index()
print(f'  Monthly complaint records: {len(complaint_monthly):,}')

N = len(df)
N_FEAT = 12
X_temporal = np.zeros((N, 12, N_FEAT), dtype=np.float32)
years = df['Year'].values
slopes = df['slope_pct'].values
su_weights = df['streetuse_weight'].values

seg_to_neigh = df['neighbourhood'].values
neigh_month_tot = {}
for _, r in complaint_monthly.iterrows():
    key = (seg_to_neigh[int(r['seg_idx'])], int(r['year']), int(r['month']))
    neigh_month_tot[key] = neigh_month_tot.get(key, 0) + r['cnt']

for i in range(N):
    yr, sl, sw = years[i], slopes[i], su_weights[i]
    wx = monthly_wx[monthly_wx['year'] == yr]
    for _, r in wx.iterrows():
        m = int(r['month']) - 1
        if 0 <= m < 12:
            X_temporal[i,m,0]  = r['avg_max_temp']
            X_temporal[i,m,1]  = r['avg_min_temp']
            X_temporal[i,m,2]  = r['total_precip_mm']
            X_temporal[i,m,3]  = r['freeze_thaw_days']
            X_temporal[i,m,11] = r['extreme_days']
            # ★ Interaction channels ★
            X_temporal[i,m,7]  = r['total_precip_mm'] * sl
            X_temporal[i,m,8]  = r['freeze_thaw_days'] * sw
            X_temporal[i,m,9]  = r['total_snow_cm'] * sl
            X_temporal[i,m,10] = (r['avg_max_temp'] - r['avg_min_temp']) * sw

    seg_c = complaint_monthly[(complaint_monthly['seg_idx']==i) & (complaint_monthly['year']==yr)]
    neigh = seg_to_neigh[i]
    for _, r in seg_c.iterrows():
        m = int(r['month']) - 1
        if 0 <= m < 12:
            X_temporal[i,m,4] = r['cnt']
            nt = neigh_month_tot.get((neigh, yr, int(r['month'])), 1)
            X_temporal[i,m,5] = min(r['cnt'] / max(nt, 1), 1.0)
            X_temporal[i,m,6] = r['pothole_cnt']

df['complaint_total'] = X_temporal[:,:,4].sum(axis=1)
df['pothole_total'] = X_temporal[:,:,6].sum(axis=1)
print(f'X_temporal: {X_temporal.shape}')

Weather: 2,557 days

Loading 311...
  Pavement-related: 135,647
  Survey-year 311: 27,187
  Snapped within 150m: 27,024
  Monthly complaint records: 22,807
X_temporal: (13764, 12, 12)


## 6. ★ CHANGED: Feature Construction ★


In [103]:
su_dummies = pd.get_dummies(df['streetuse'], prefix='su')
for c in ['su_Arterial','su_Collector','su_Residential']:
    if c not in su_dummies.columns: su_dummies[c] = 0

# ★ Road-MLP: 13 dims (restored features + spatial lag + neigh encoding) ★
X_road_raw = np.column_stack([
    df['length_(m)'].fillna(df['length_(m)'].median()).values,
    su_dummies[['su_Arterial','su_Collector','su_Residential']].values,
    df['elevation_m'].values,
    df['slope_pct'].values,
    df['repair_count'].values,              # ★ restored from v3
    df['segment_density'].values,           # ★ restored from v3
    df['source'].values,                    # ★ restored from v3
    df['dist_to_traffic_count'].values,     # ★ restored from v3
    df['spatial_lag_risk'].values,          # ★ NEW
    df['spatial_lag_high'].values,          # ★ NEW
    df['neigh_high_pct'].values,            # ★ NEW (target encoding)
]).astype(np.float32)

# ★ Tabular-MLP: ~35 dims (original + spatial lag + neigh encoding) ★
neigh_dummies = pd.get_dummies(df['neighbourhood'], prefix='n')
X_tab_raw = np.column_stack([
    neigh_dummies.values,
    df['ROW_width'].values,
    df['complaint_total'].values,
    df['pothole_total'].values,
    df['repair_count'].values,
    df['segment_density'].values,
    df['dist_to_traffic_count'].values,
    df['spatial_lag_risk'].values,           # ★ NEW
    df['spatial_lag_high'].values,           # ★ NEW
    df['spatial_lag_med'].values,            # ★ NEW
    df['neigh_low_pct'].values,              # ★ NEW (target encoding)
    df['neigh_med_pct'].values,              # ★ NEW
    df['neigh_high_pct'].values,             # ★ NEW
]).astype(np.float32)

print(f'Road-MLP:    {X_road_raw.shape[1]} dims  (6 physical + 4 restored + 3 spatial/neigh)')
print(f'Tabular-MLP: {X_tab_raw.shape[1]} dims  (neigh*{len(neigh_dummies.columns)} + ROW + complaints + repair + density + traffic + 6 new)')

Road-MLP:    13 dims  (6 physical + 4 restored + 3 spatial/neigh)
Tabular-MLP: 35 dims  (neigh*23 + ROW + complaints + repair + density + traffic + 6 new)


## 7. Normalize + ★ RandomOverSampler ★


In [104]:
sc_road = StandardScaler().fit(X_road_raw[idx_train])
sc_tab  = StandardScaler().fit(X_tab_raw[idx_train])
sc_temp = StandardScaler().fit(X_temporal[idx_train].reshape(-1, N_FEAT))

def split_norm(X, sc, idxs, seq=False, nf=None):
    if seq:
        n = len(idxs)
        return sc.transform(X[idxs].reshape(-1, nf)).reshape(n, 12, nf).astype(np.float32)
    return sc.transform(X[idxs]).astype(np.float32)

Xr_tr = split_norm(X_road_raw, sc_road, idx_train)
Xr_v  = split_norm(X_road_raw, sc_road, idx_val)
Xr_te = split_norm(X_road_raw, sc_road, idx_test)
Xt_tr = split_norm(X_temporal, sc_temp, idx_train, True, N_FEAT)
Xt_v  = split_norm(X_temporal, sc_temp, idx_val, True, N_FEAT)
Xt_te = split_norm(X_temporal, sc_temp, idx_test, True, N_FEAT)
Xb_tr = split_norm(X_tab_raw, sc_tab, idx_train)
Xb_v  = split_norm(X_tab_raw, sc_tab, idx_val)
Xb_te = split_norm(X_tab_raw, sc_tab, idx_test)
y_tr, y_v, y_te = y[idx_train], y[idx_val], y[idx_test]

# ★ Save ORIGINAL (pre-ROS) data for XGBoost/LightGBM ★
# Tree models handle imbalance internally via class_weight — ROS hurts them
Xr_tr_orig, Xt_tr_orig, Xb_tr_orig, y_tr_orig = Xr_tr.copy(), Xt_tr.copy(), Xb_tr.copy(), y_tr.copy()

# ★ NEW: RandomOverSampler for minority class balancing (neural nets only) ★
try:
    from imblearn.over_sampling import RandomOverSampler
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'])
    from imblearn.over_sampling import RandomOverSampler

print(f'\nBefore ROS: {np.bincount(y_tr)}')
flat_tr = np.hstack([Xr_tr, Xt_tr.reshape(len(Xr_tr), -1), Xb_tr])
ros = RandomOverSampler(random_state=42)
_, _ = ros.fit_resample(flat_tr, y_tr)
res_idx = ros.sample_indices_
Xr_tr = Xr_tr[res_idx]
Xt_tr = Xt_tr[res_idx]
Xb_tr = Xb_tr[res_idx]
y_tr  = y_tr[res_idx]
print(f'After  ROS: {np.bincount(y_tr)}')

# Class weights (recompute after ROS)
class_counts = np.bincount(y_tr, minlength=NUM_CLASSES)
cw = 1.0 / class_counts
cw = cw / cw.sum() * NUM_CLASSES
CW = torch.FloatTensor(cw).to(DEVICE)
print(f'Class weights: {dict(zip(CLASS_NAMES, [f"{w:.3f}" for w in cw]))}')
print(f'\nRoad: {Xr_tr.shape}, Temporal: {Xt_tr.shape}, Tabular: {Xb_tr.shape}')


Before ROS: [2273 4125 3236]
After  ROS: [4125 4125 4125]
Class weights: {'Low': '1.000', 'Medium': '1.000', 'High': '1.000'}

Road: (12375, 13), Temporal: (12375, 12, 12), Tabular: (12375, 35)


## 8. Loss Functions


In [105]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha, self.gamma, self.label_smoothing = alpha, gamma, label_smoothing
    def forward(self, logits, targets):
        nc = logits.size(1)
        if self.label_smoothing > 0:
            with torch.no_grad():
                smooth = torch.full_like(logits, self.label_smoothing / (nc - 1))
                smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
            loss = -(smooth * torch.log_softmax(logits, dim=1)).sum(dim=1)
        else:
            loss = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.softmax(logits, dim=1).gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = (1 - p_t) ** self.gamma * loss
        if self.alpha is not None: loss = self.alpha[targets] * loss
        return loss.mean()

class OrdinalPenalty(nn.Module):
    def __init__(self, weight=0.3):
        super().__init__()
        self.weight = weight
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        classes = torch.arange(logits.size(1), device=logits.device, dtype=torch.float32)
        expected = (probs * classes.unsqueeze(0)).sum(dim=1)
        return self.weight * ((expected - targets.float()) ** 2).mean()

## 9. Branch Models


In [106]:
# ★ Road-MLP: 13d -> 16d embedding ★
class RoadMLP(nn.Module):
    def __init__(self, dim, emb=16, hidden=64, drop=0.4):
        super().__init__()
        self.emb_dim = emb
        self.enc = nn.Sequential(
            nn.Linear(dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, emb), nn.ReLU())
        self.head = nn.Linear(emb, NUM_CLASSES)
    def get_embedding(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.enc(x))

# TemporalCNN: 12ch -> 32d embedding (unchanged)
class TemporalCNN(nn.Module):
    def __init__(self, in_channels=12, emb=32, drop=0.4):
        super().__init__()
        self.emb_dim = emb
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(drop),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(drop),
            nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Sequential(nn.Linear(64, emb), nn.ReLU(), nn.Dropout(drop))
        self.head = nn.Linear(emb, NUM_CLASSES)
    def get_embedding(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x).squeeze(-1)
        return self.fc(x)
    def forward(self, x): return self.head(self.get_embedding(x))

# Tabular-MLP: ~35d -> 16d embedding
class TabularMLP(nn.Module):
    def __init__(self, dim, emb=16, hidden=128, drop=0.4):
        super().__init__()
        self.emb_dim = emb
        self.enc = nn.Sequential(
            nn.Linear(dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, emb), nn.ReLU())
        self.head = nn.Linear(emb, NUM_CLASSES)
    def get_embedding(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.enc(x))

## 10. Dataset


In [107]:
class FusionDS(Dataset):
    def __init__(self, xr, xt, xb, y):
        self.xr, self.xt, self.xb, self.y = xr, xt, xb, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.xr[i], self.xt[i], self.xb[i], self.y[i]

## 11. ★ CHANGED: Slimmer Cross-Attention Fusion (d_model=32) ★


In [108]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, road, temporal, tabular, d_model=32, n_heads=4, drop=0.3):
        super().__init__()
        self.road, self.temporal, self.tabular = road, temporal, tabular

        self.proj_road = nn.Linear(road.emb_dim, d_model)
        self.proj_temp = nn.Linear(temporal.emb_dim, d_model)
        self.proj_tab  = nn.Linear(tabular.emb_dim, d_model)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads, dropout=drop, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.attn_ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(d_model * 2, d_model))
        self.ffn_norm = nn.LayerNorm(d_model)

        self.gate = nn.Sequential(
            nn.Linear(d_model * 3, 32), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(32, 3))

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(32, NUM_CLASSES))

    def _get_projections(self, xr, xt, xb):
        er = self.proj_road(self.road.get_embedding(xr))
        et = self.proj_temp(self.temporal.get_embedding(xt))
        eb = self.proj_tab(self.tabular.get_embedding(xb))
        return er, et, eb

    def _cross_attend(self, er, et, eb):
        seq = torch.stack([er, et, eb], dim=1)
        attn_out, _ = self.cross_attn(seq, seq, seq)
        seq = self.attn_norm(seq + attn_out)
        ffn_out = self.attn_ffn(seq)
        seq = self.ffn_norm(seq + ffn_out)
        return seq[:, 0], seq[:, 1], seq[:, 2]

    def get_gate_weights(self, xr, xt, xb):
        with torch.no_grad():
            er, et, eb = self._get_projections(xr, xt, xb)
            er, et, eb = self._cross_attend(er, et, eb)
            cat = torch.cat([er, et, eb], dim=1)
            return torch.softmax(self.gate(cat), dim=1)

    def forward(self, xr, xt, xb):
        er, et, eb = self._get_projections(xr, xt, xb)
        er, et, eb = self._cross_attend(er, et, eb)
        cat = torch.cat([er, et, eb], dim=1)
        weights = torch.softmax(self.gate(cat), dim=1)
        fused = weights[:,0:1]*er + weights[:,1:2]*et + weights[:,2:3]*eb
        return self.classifier(fused)

    def forward_ablation(self, xr, xt, xb, disable=None):
        er = self.proj_road(self.road.get_embedding(xr))
        et = self.proj_temp(self.temporal.get_embedding(xt))
        eb = self.proj_tab(self.tabular.get_embedding(xb))
        if disable == 'road': er = torch.zeros_like(er)
        elif disable == 'temporal': et = torch.zeros_like(et)
        elif disable == 'tabular': eb = torch.zeros_like(eb)
        er, et, eb = self._cross_attend(er, et, eb)
        cat = torch.cat([er, et, eb], dim=1)
        weights = torch.softmax(self.gate(cat), dim=1)
        fused = weights[:,0:1]*er + weights[:,1:2]*et + weights[:,2:3]*eb
        return self.classifier(fused)

print(f'Models defined. NUM_CLASSES={NUM_CLASSES}')
print(f'CrossAttentionFusion: d_model=32, n_heads=4 (slimmer than v6)')

Models defined. NUM_CLASSES=3
CrossAttentionFusion: d_model=32, n_heads=4 (slimmer than v6)


## 12. Training Utilities


In [109]:
def train_branch(model, X_tr, X_v, y_tr_, y_v_, epochs=60, bs=256, lr=2e-3, patience=12, verbose=True):
    model = model.to(DEVICE)
    tr_dl = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr_).long()), batch_size=bs, shuffle=True)
    v_dl  = DataLoader(TensorDataset(torch.tensor(X_v), torch.tensor(y_v_).long()), batch_size=bs)
    focal = FocalLoss(alpha=CW, gamma=2.0, label_smoothing=0.1)
    ordinal_loss = OrdinalPenalty(weight=0.3)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, wait, best_sd = 0, 0, None
    for ep in range(1, epochs+1):
        model.train()
        for X, yy in tr_dl:
            X, yy = X.to(DEVICE), yy.to(DEVICE)
            logits = model(X)
            loss = focal(logits, yy) + ordinal_loss(logits, yy)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for X, yy in v_dl:
                preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
                labs.extend(yy.numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: wait += 1
        if verbose and ep % 10 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}')
        if wait >= patience:
            if verbose: print(f'  Early stop ep {ep}')
            break
    model.load_state_dict(best_sd)
    return model, best_f1

def eval_test(model, X_te, y_te_, bs=256):
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te_).long()), batch_size=bs)
    preds, labs = [], []
    with torch.no_grad():
        for X, yy in dl:
            preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            labs.extend(yy.numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs)

print('Training utilities ready.')

Training utilities ready.


## 13. Train Branches


In [110]:
print('=== Road-MLP (13d -> 16d) ===')
road_model = RoadMLP(dim=Xr_tr.shape[1])
road_model, road_val = train_branch(road_model, Xr_tr, Xr_v, y_tr, y_v)
road_test, _, _ = eval_test(road_model, Xr_te, y_te)
print(f'  val={road_val:.4f} test={road_test:.4f}\n')

print('=== TemporalCNN (12ch -> 32d) ===')
temp_model = TemporalCNN(in_channels=N_FEAT, emb=32)
temp_model, temp_val = train_branch(temp_model, Xt_tr, Xt_v, y_tr, y_v)
temp_test, _, _ = eval_test(temp_model, Xt_te, y_te)
print(f'  val={temp_val:.4f} test={temp_test:.4f}\n')

print('=== Tabular-MLP (~35d -> 16d) ===')
tab_model = TabularMLP(dim=Xb_tr.shape[1])
tab_model, tab_val = train_branch(tab_model, Xb_tr, Xb_v, y_tr, y_v)
tab_test, _, _ = eval_test(tab_model, Xb_te, y_te)
print(f'  val={tab_val:.4f} test={tab_test:.4f}\n')

print(f'{"Branch":<15} {"Val":>6} {"Test":>6}')
print('-'*29)
for n,v,t in [('Road-MLP',road_val,road_test),('TemporalCNN',temp_val,temp_test),('Tabular-MLP',tab_val,tab_test)]:
    print(f'{n:<15} {v:>6.4f} {t:>6.4f}')

=== Road-MLP (13d -> 16d) ===
  Ep  10 val_f1=0.4956 best=0.5110
  Ep  20 val_f1=0.5058 best=0.5128
  Ep  30 val_f1=0.5087 best=0.5159
  Early stop ep 39
  val=0.5159 test=0.5057

=== TemporalCNN (12ch -> 32d) ===
  Ep  10 val_f1=0.3861 best=0.3924
  Ep  20 val_f1=0.3797 best=0.3924
  Early stop ep 20
  val=0.3924 test=0.3987

=== Tabular-MLP (~35d -> 16d) ===
  Ep  10 val_f1=0.5029 best=0.5083
  Ep  20 val_f1=0.5037 best=0.5141
  Ep  30 val_f1=0.5112 best=0.5142
  Early stop ep 37
  val=0.5142 test=0.5016

Branch             Val   Test
-----------------------------
Road-MLP        0.5159 0.5057
TemporalCNN     0.3924 0.3987
Tabular-MLP     0.5142 0.5016


## 14. ★ Cross-Attention Fusion (d_model=32) ★


In [111]:
fusion = CrossAttentionFusion(road_model, temp_model, tab_model, d_model=32, n_heads=4).to(DEVICE)
print(f'\nFusion params: {sum(p.numel() for p in fusion.parameters()):,}  (vs v6: 74,831)')

tr_dl = DataLoader(FusionDS(torch.tensor(Xr_tr),torch.tensor(Xt_tr),torch.tensor(Xb_tr),torch.tensor(y_tr).long()), batch_size=256, shuffle=True)
v_dl  = DataLoader(FusionDS(torch.tensor(Xr_v),torch.tensor(Xt_v),torch.tensor(Xb_v),torch.tensor(y_v).long()), batch_size=256)
te_dl = DataLoader(FusionDS(torch.tensor(Xr_te),torch.tensor(Xt_te),torch.tensor(Xb_te),torch.tensor(y_te).long()), batch_size=256)

focal = FocalLoss(alpha=CW, gamma=2.0, label_smoothing=0.1)
ordinal = OrdinalPenalty(weight=0.3)

def eval_fusion(model, dl, disable=None):
    model.eval()
    preds, labs, all_probs = [], [], []
    with torch.no_grad():
        for xr,xt,xb,yy in dl:
            xr,xt,xb = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE)
            logits = model.forward_ablation(xr,xt,xb,disable) if disable else model(xr,xt,xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            preds.extend(logits.argmax(1).cpu().numpy()); labs.extend(yy.numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs), np.vstack(all_probs)

# Phase 1: Warmup (freeze branches)
print('\n--- Phase 1: Warmup (5 ep, branches frozen) ---')
for p in list(fusion.road.parameters()) + list(fusion.temporal.parameters()) + list(fusion.tabular.parameters()):
    p.requires_grad = False
opt = torch.optim.AdamW([p for p in fusion.parameters() if p.requires_grad], lr=2e-3, weight_decay=1e-4)
for ep in range(1, 6):
    fusion.train()
    for xr,xt,xb,yy in tr_dl:
        xr,xt,xb,yy = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE),yy.to(DEVICE)
        logits = fusion(xr,xt,xb)
        loss = focal(logits, yy) + ordinal(logits, yy)
        opt.zero_grad(); loss.backward(); opt.step()
    f1, _, _, _ = eval_fusion(fusion, v_dl)
    print(f'  Warmup {ep}/5 val_f1={f1:.4f}')

# Phase 2: End-to-end
print('\n--- Phase 2: End-to-end (60 ep) ---')
for p in fusion.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(fusion.parameters(), lr=2e-4, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)
best_f1, wait, best_sd = 0, 0, None
for ep in range(1, 61):
    fusion.train()
    for xr,xt,xb,yy in tr_dl:
        xr,xt,xb,yy = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE),yy.to(DEVICE)
        logits = fusion(xr,xt,xb)
        loss = focal(logits, yy) + ordinal(logits, yy)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(fusion.parameters(), 1.0); opt.step()
    sch.step()
    f1, _, _, _ = eval_fusion(fusion, v_dl)
    mk = ''
    if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in fusion.state_dict().items()}; mk = ' *'
    else: wait += 1
    if ep % 5 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}{mk}')
    if wait >= 12: print(f'  Early stop ep {ep}'); break
fusion.load_state_dict(best_sd); fusion = fusion.to(DEVICE)

test_f1, preds, labs, test_probs_fusion = eval_fusion(fusion, te_dl)
print(f'\n{"="*60}')
print(f'CROSS-ATTENTION FUSION TEST MACRO F1: {test_f1:.4f}')
print(f'{"="*60}')
print(classification_report(labs, preds, target_names=CLASS_NAMES))

cm = confusion_matrix(labs, preds)
print('Confusion Matrix:')
header = ''.join(f'{"Pred "+n:>16}' for n in CLASS_NAMES)
print(f'{"":>16}{header}')
for i, name in enumerate(CLASS_NAMES):
    row = ''.join(f'{cm[i,j]:>16}' for j in range(NUM_CLASSES))
    print(f'{"True "+name:>16}{row}')

# Ablation
print(f'\n{"="*60}')
print('ABLATION STUDY')
print(f'{"="*60}')
results = {'Full model': test_f1}
for branch in ['road','temporal','tabular']:
    ab_f1, _, _, _ = eval_fusion(fusion, te_dl, disable=branch)
    results[f'w/o {branch}'] = ab_f1
print(f'\n{"Config":<20} {"F1":>6} {"Delta":>8}')
print('-'*36)
for k, v in results.items():
    d = f'{v - test_f1:+.4f}' if k != 'Full model' else '  base'
    print(f'{k:<20} {v:>6.4f} {d:>8}')

# Gate weights
print(f'\n{"="*60}')
print('GATE WEIGHT ANALYSIS')
print(f'{"="*60}')
all_w = []
with torch.no_grad():
    for xr,xt,xb,yy in te_dl:
        all_w.append(fusion.get_gate_weights(xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE)).cpu().numpy())
all_w = np.vstack(all_w)
print(f'Mean gate weights:')
print(f'  Road:     {all_w[:,0].mean():.4f} +/- {all_w[:,0].std():.4f}')
print(f'  Temporal: {all_w[:,1].mean():.4f} +/- {all_w[:,1].std():.4f}')
print(f'  Tabular:  {all_w[:,2].mean():.4f} +/- {all_w[:,2].std():.4f}')
for cls, name in enumerate(CLASS_NAMES):
    mask_cls = labs == cls
    if mask_cls.sum() > 0:
        print(f'  {name}: R={all_w[mask_cls,0].mean():.3f} T={all_w[mask_cls,1].mean():.3f} Tab={all_w[mask_cls,2].mean():.3f}')


Fusion params: 42,703  (vs v6: 74,831)

--- Phase 1: Warmup (5 ep, branches frozen) ---
  Warmup 1/5 val_f1=0.5008
  Warmup 2/5 val_f1=0.5034
  Warmup 3/5 val_f1=0.5103
  Warmup 4/5 val_f1=0.5137
  Warmup 5/5 val_f1=0.5211

--- Phase 2: End-to-end (60 ep) ---
  Ep   5 val_f1=0.5079 best=0.5177
  Ep  10 val_f1=0.5092 best=0.5177
  Early stop ep 13

CROSS-ATTENTION FUSION TEST MACRO F1: 0.5055
              precision    recall  f1-score   support

         Low       0.45      0.54      0.49       487
      Medium       0.55      0.43      0.48       885
        High       0.51      0.58      0.54       693

    accuracy                           0.51      2065
   macro avg       0.50      0.52      0.51      2065
weighted avg       0.51      0.51      0.50      2065

Confusion Matrix:
                        Pred Low     Pred Medium       Pred High
        True Low             262             142              83
     True Medium             198             379             308
       Tru

## 15. ★ XGBoost + LightGBM ★


In [112]:
try:
    from xgboost import XGBClassifier
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    from xgboost import XGBClassifier

try:
    import lightgbm as lgb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'])
    import lightgbm as lgb

# ★ Use ORIGINAL (non-oversampled) data for tree models ★
X_flat_tr = np.concatenate([Xr_tr_orig, Xt_tr_orig.reshape(len(Xr_tr_orig),-1), Xb_tr_orig], axis=1)
X_flat_v  = np.concatenate([Xr_v,  Xt_v.reshape(len(Xr_v),-1),  Xb_v],  axis=1)
X_flat_te = np.concatenate([Xr_te, Xt_te.reshape(len(Xr_te),-1), Xb_te], axis=1)

# --- XGBoost (on original data, NOT oversampled) ---
print('\n--- XGBoost (original data) ---')
xgb = XGBClassifier(n_estimators=500, max_depth=7, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                     eval_metric='mlogloss', num_class=NUM_CLASSES,
                     random_state=42, verbosity=0)
xgb.fit(X_flat_tr, y_tr_orig, eval_set=[(X_flat_v, y_v)], verbose=False)
xgb_preds = xgb.predict(X_flat_te)
xgb_f1 = f1_score(y_te, xgb_preds, average='macro')
print(f'XGBoost Test Macro F1: {xgb_f1:.4f}')
print(classification_report(y_te, xgb_preds, target_names=CLASS_NAMES))

# --- ★ NEW: LightGBM (on original data, NOT oversampled) ---
print('--- LightGBM (original data) ---')
lgb_cw = {c: len(y_tr_orig) / (3 * (y_tr_orig == c).sum()) for c in range(3)}
lgb_model = lgb.LGBMClassifier(
    n_estimators=800, max_depth=7, learning_rate=0.03,
    num_leaves=63, subsample=0.8, colsample_bytree=0.7,
    class_weight=lgb_cw, random_state=42, verbose=-1)
lgb_model.fit(X_flat_tr, y_tr_orig,
              eval_set=[(X_flat_v, y_v)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
lgb_preds = lgb_model.predict(X_flat_te)
lgb_f1 = f1_score(y_te, lgb_preds, average='macro')
print(f'LightGBM Test Macro F1: {lgb_f1:.4f}')
print(classification_report(y_te, lgb_preds, target_names=CLASS_NAMES))


--- XGBoost (original data) ---
XGBoost Test Macro F1: 0.5089
              precision    recall  f1-score   support

         Low       0.55      0.38      0.45       487
      Medium       0.51      0.63      0.57       885
        High       0.54      0.48      0.51       693

    accuracy                           0.52      2065
   macro avg       0.53      0.50      0.51      2065
weighted avg       0.53      0.52      0.52      2065

--- LightGBM (original data) ---
LightGBM Test Macro F1: 0.5112
              precision    recall  f1-score   support

         Low       0.50      0.51      0.51       487
      Medium       0.54      0.47      0.50       885
        High       0.49      0.57      0.53       693

    accuracy                           0.51      2065
   macro avg       0.51      0.52      0.51      2065
weighted avg       0.51      0.51      0.51      2065



## 16. ★ CHANGED: 3-Model Ensemble (weights tuned on VAL, not test) ★


In [113]:
print(f'\n{"="*60}')
print('3-MODEL ENSEMBLE')
print(f'{"="*60}')

# Get validation probs for weight tuning
_, _, _, val_probs_fusion = eval_fusion(fusion, v_dl)
val_probs_xgb = xgb.predict_proba(X_flat_v)
val_probs_lgb = lgb_model.predict_proba(X_flat_v)

# Test probs
xgb_probs_te = xgb.predict_proba(X_flat_te)
lgb_probs_te = lgb_model.predict_proba(X_flat_te)

# ★ Grid search on VALIDATION set (not test!) ★
best_weights, best_val_f1 = None, 0
for wf in np.arange(0.05, 0.60, 0.05):
    for wl in np.arange(0.05, 0.60, 0.05):
        wx = 1.0 - wf - wl
        if wx < 0.05: continue
        ens_p = wf * val_probs_fusion + wx * val_probs_xgb + wl * val_probs_lgb
        ens_pred = ens_p.argmax(axis=1)
        ef = f1_score(y_v, ens_pred, average='macro')
        if ef > best_val_f1:
            best_weights, best_val_f1 = (wf, wx, wl), ef

wf, wx, wl = best_weights
print(f'Best weights (tuned on val): Fusion={wf:.2f}, XGB={wx:.2f}, LGB={wl:.2f}')
print(f'Val F1 with best weights: {best_val_f1:.4f}')

# Apply to test
ens_probs_te = wf * test_probs_fusion + wx * xgb_probs_te + wl * lgb_probs_te
ens_preds = ens_probs_te.argmax(axis=1)
ens_f1 = f1_score(y_te, ens_preds, average='macro')
print(f'\n3-Model Ensemble Test Macro F1: {ens_f1:.4f}')
print(classification_report(y_te, ens_preds, target_names=CLASS_NAMES))

# Also report 2-model ensemble for comparison
best_2w, best_2f = 0, 0
for w in np.arange(0.1, 0.9, 0.05):
    ens_p = w * val_probs_fusion + (1-w) * val_probs_xgb
    ef = f1_score(y_v, ens_p.argmax(1), average='macro')
    if ef > best_2f: best_2w, best_2f = w, ef
ens_2_te = best_2w * test_probs_fusion + (1-best_2w) * xgb_probs_te
ens_2_f1 = f1_score(y_te, ens_2_te.argmax(1), average='macro')
print(f'\n2-Model Ensemble (Fusion+XGB) Test F1: {ens_2_f1:.4f} (fusion_w={best_2w:.2f})')


3-MODEL ENSEMBLE
Best weights (tuned on val): Fusion=0.25, XGB=0.30, LGB=0.45
Val F1 with best weights: 0.5393

3-Model Ensemble Test Macro F1: 0.5183
              precision    recall  f1-score   support

         Low       0.52      0.47      0.50       487
      Medium       0.52      0.55      0.54       885
        High       0.52      0.52      0.52       693

    accuracy                           0.52      2065
   macro avg       0.52      0.51      0.52      2065
weighted avg       0.52      0.52      0.52      2065


2-Model Ensemble (Fusion+XGB) Test F1: 0.5193 (fusion_w=0.80)


## 17. ★ NEW: Per-Class Threshold Optimization ★


In [114]:
from scipy.optimize import differential_evolution

print(f'\n{"="*60}')
print('PER-CLASS THRESHOLD OPTIMIZATION')
print(f'{"="*60}')

def apply_thresholds(probs, t):
    adjusted = probs.copy()
    for c, tc in enumerate(t):
        adjusted[:, c] = adjusted[:, c] - tc
    return adjusted.argmax(1)

def neg_macro_f1(t, probs, y_true):
    return -f1_score(y_true, apply_thresholds(probs, t), average='macro')

# Optimize on validation ensemble probs
ens_probs_v = wf * val_probs_fusion + wx * val_probs_xgb + wl * val_probs_lgb
result = differential_evolution(
    neg_macro_f1, [(-0.5, 0.5)] * 3,
    args=(ens_probs_v, y_v), seed=42, maxiter=300, tol=1e-6)
opt_thresholds = result.x
opt_val_f1 = -result.fun

print(f'Optimal thresholds: {[f"{t:.4f}" for t in opt_thresholds]}')
print(f'Val F1 with thresholds: {opt_val_f1:.4f} (vs {best_val_f1:.4f} without)')

# Apply to test
tuned_preds = apply_thresholds(ens_probs_te, opt_thresholds)
tuned_f1 = f1_score(y_te, tuned_preds, average='macro')
print(f'\nThreshold-Tuned Test Macro F1: {tuned_f1:.4f}')
print(classification_report(y_te, tuned_preds, target_names=CLASS_NAMES))

# Per-class comparison
print('Per-class F1 comparison:')
print(f'  {"Class":<10} {"Argmax":>8} {"Tuned":>8} {"Delta":>8}')
for c, name in enumerate(CLASS_NAMES):
    f_arg = f1_score((y_te==c).astype(int), (ens_preds==c).astype(int))
    f_tun = f1_score((y_te==c).astype(int), (tuned_preds==c).astype(int))
    print(f'  {name:<10} {f_arg:>8.4f} {f_tun:>8.4f} {f_tun-f_arg:>+8.4f}')


PER-CLASS THRESHOLD OPTIMIZATION
Optimal thresholds: ['0.1359', '0.0925', '0.0701']
Val F1 with thresholds: 0.5454 (vs 0.5393 without)

Threshold-Tuned Test Macro F1: 0.5157
              precision    recall  f1-score   support

         Low       0.55      0.43      0.48       487
      Medium       0.52      0.54      0.53       885
        High       0.51      0.57      0.54       693

    accuracy                           0.52      2065
   macro avg       0.53      0.51      0.52      2065
weighted avg       0.52      0.52      0.52      2065

Per-class F1 comparison:
  Class        Argmax    Tuned    Delta
  Low          0.4951   0.4850  -0.0101
  Medium       0.5384   0.5267  -0.0117
  High         0.5214   0.5355  +0.0142


## 18. SHAP Analysis


In [115]:
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'shap'])
    import shap

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fusion.eval()

R_DIM = Xr_tr.shape[1]
T_FLAT = Xt_tr.reshape(len(Xr_tr),-1).shape[1]
B_DIM = Xb_tr.shape[1]

def model_predict_flat(flat_input):
    t = torch.FloatTensor(flat_input).to(DEVICE)
    xr = t[:, :R_DIM]
    xt = t[:, R_DIM:R_DIM+T_FLAT].reshape(-1, 12, N_FEAT)
    xb = t[:, R_DIM+T_FLAT:]
    with torch.no_grad():
        return torch.softmax(fusion(xr, xt, xb), dim=1).cpu().numpy()

bg = np.hstack([Xr_tr[:100], Xt_tr[:100].reshape(100,-1), Xb_tr[:100]])
test_flat = np.hstack([Xr_te[:200], Xt_te[:200].reshape(200,-1), Xb_te[:200]])

print('\nRunning SHAP KernelExplainer (this may take a few minutes)...')
explainer = shap.KernelExplainer(model_predict_flat, bg)
shap_values = explainer.shap_values(test_flat, nsamples=100)

# Feature names
road_names = ['length','su_Arterial','su_Collector','su_Residential','elevation','slope',
              'repair_count','segment_density','source','dist_to_traffic',
              'spatial_lag_risk','spatial_lag_high','neigh_high_pct']
temp_names = [f'm{m+1}_{f}' for m in range(12) for f in ['max_t','min_t','precip','freeze','complaint','density','pothole','precip_slope','freeze_street','snow_slope','range_street','extreme']]
neigh_cols = list(neigh_dummies.columns)
tab_names = neigh_cols + ['ROW_width','complaint_total','pothole_total','repair_count','segment_density','dist_to_traffic',
                          'spatial_lag_risk','spatial_lag_high','spatial_lag_med',
                          'neigh_low_pct','neigh_med_pct','neigh_high_pct']
all_names = road_names + temp_names + tab_names

if isinstance(shap_values, list):
    sv = shap_values[2]  # class 2 = High
else:
    sv = shap_values
    if sv.ndim == 3:
        sv = sv[:, :, 2]

mean_abs = np.abs(sv).mean(axis=0)

# Branch-level importance
road_imp = mean_abs[:R_DIM].sum()
temp_imp = mean_abs[R_DIM:R_DIM+T_FLAT].sum()
tab_imp = mean_abs[R_DIM+T_FLAT:].sum()
total_imp = road_imp + temp_imp + tab_imp

print(f'\n{"="*55}')
print('SHAP Branch-Level Importance (High-risk class)')
print(f'{"="*55}')
print(f'  Road-MLP:     {road_imp/total_imp*100:.1f}%')
print(f'  TemporalCNN:  {temp_imp/total_imp*100:.1f}%')
print(f'  Tabular-MLP:  {tab_imp/total_imp*100:.1f}%')

top_idx = np.argsort(mean_abs)[::-1][:15]
print(f'\nTop 15 features by mean |SHAP|:')
for i, idx in enumerate(top_idx):
    idx = int(idx)
    name = all_names[idx] if idx < len(all_names) else f'feature_{idx}'
    print(f'  {i+1:>2}. {name:<30} {mean_abs[idx]:.4f}')

fig, ax = plt.subplots(figsize=(10, 6))
top_names_plot = [all_names[int(i)] if int(i) < len(all_names) else f'f_{i}' for i in top_idx[:15]]
top_vals = [mean_abs[int(i)] for i in top_idx[:15]]
ax.barh(range(15), top_vals[::-1])
ax.set_yticks(range(15))
ax.set_yticklabels(top_names_plot[::-1])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('CityBrain v7 — Top 15 Feature Importance (High Risk)')
plt.tight_layout()
plt.savefig('shap_importance_v7.png', dpi=150)
plt.show()
print('Saved shap_importance_v7.png')


Running SHAP KernelExplainer (this may take a few minutes)...


  0%|          | 0/200 [00:00<?, ?it/s]


SHAP Branch-Level Importance (High-risk class)
  Road-MLP:     37.5%
  TemporalCNN:  24.2%
  Tabular-MLP:  38.3%

Top 15 features by mean |SHAP|:
   1. spatial_lag_risk               0.0310
   2. spatial_lag_risk               0.0210
   3. spatial_lag_high               0.0155
   4. source                         0.0120
   5. length                         0.0055
   6. spatial_lag_high               0.0040
   7. pothole_total                  0.0025
   8. spatial_lag_med                0.0024
   9. n_Shaughnessy                  0.0018
  10. segment_density                0.0012
  11. m8_max_t                       0.0011
  12. m2_precip_slope                0.0011
  13. segment_density                0.0011
  14. n_Victoria-Fraserview          0.0010
  15. dist_to_traffic                0.0010
Saved shap_importance_v7.png


## 19. FINAL COMPARISON


In [116]:
# Determine best result
best_model_name = 'Threshold-Tuned 3-Model Ensemble'
best_test_f1 = tuned_f1

print(f'\n{"="*60}')
print('FINAL COMPARISON — ALL VERSIONS')
print(f'{"="*60}')
print(f'{"Model":<50} {"Test Macro F1":>12}')
print('-'*64)
print(f'{"v1 Baseline (old 3-class, concat fusion)":<50} {"0.3915":>12}')
print(f'{"v2 Enhanced (old 3-class)":<50} {"0.4358":>12}')
print(f'{"v5 GatedFusion (rebalanced)":<50} {"0.4241":>12}')
print(f'{"v6 CrossAttention Fusion":<50} {"0.4400":>12}')
print(f'{"v6 Simple Ensemble (Fusion+XGB)":<50} {"0.5009":>12}')
print(f'---')
print(f'{"v7 Road-MLP (13d) branch":<50} {road_test:>12.4f}')
print(f'{"v7 TemporalCNN branch":<50} {temp_test:>12.4f}')
print(f'{"v7 Tabular-MLP (~35d) branch":<50} {tab_test:>12.4f}')
print(f'{"v7 CrossAttention Fusion (d_model=32)":<50} {test_f1:>12.4f}')
print(f'{"v7 XGBoost":<50} {xgb_f1:>12.4f}')
print(f'{"v7 LightGBM":<50} {lgb_f1:>12.4f}')
print(f'{"v7 2-Model Ensemble (Fusion+XGB)":<50} {ens_2_f1:>12.4f}')
print(f'{"v7 3-Model Ensemble (Fusion+XGB+LGB)":<50} {ens_f1:>12.4f}')
print(f'{"★ v7 Threshold-Tuned 3-Model Ensemble":<50} {tuned_f1:>12.4f}')
print(f'\n{"="*60}')
print(f'BEST: {best_model_name} = {best_test_f1:.4f}')
print(f'Improvement over v6 best (0.5009): {best_test_f1 - 0.5009:+.4f}')
print(f'{"="*60}')

print(f'\nAll {len(df):,} samples used.')
print(f'Label: VERY GOOD->Low, GOOD+FAIR->Medium, POOR+VERY POOR->High')
print(f'\nv7 changes:')
print(f'  1. Spatial lag features (7-NN, train-only)')
print(f'  2. Neighbourhood target encoding (train-only)')
print(f'  3. Road-MLP 6d -> 13d (restored + spatial + neigh)')
print(f'  4. Tabular-MLP ~29d -> ~35d (+ spatial + neigh encoding)')
print(f'  5. RandomOverSampler for class balance')
print(f'  6. Slimmer CrossAttention (d_model=32, params halved)')
print(f'  7. LightGBM as 3rd ensemble member')
print(f'  8. Ensemble weights tuned on VAL (not test)')
print(f'  9. Per-class threshold optimization')


FINAL COMPARISON — ALL VERSIONS
Model                                              Test Macro F1
----------------------------------------------------------------
v1 Baseline (old 3-class, concat fusion)                 0.3915
v2 Enhanced (old 3-class)                                0.4358
v5 GatedFusion (rebalanced)                              0.4241
v6 CrossAttention Fusion                                 0.4400
v6 Simple Ensemble (Fusion+XGB)                          0.5009
---
v7 Road-MLP (13d) branch                                 0.5057
v7 TemporalCNN branch                                    0.3987
v7 Tabular-MLP (~35d) branch                             0.5016
v7 CrossAttention Fusion (d_model=32)                    0.5055
v7 XGBoost                                               0.5089
v7 LightGBM                                              0.5112
v7 2-Model Ensemble (Fusion+XGB)                         0.5193
v7 3-Model Ensemble (Fusion+XGB+LGB)                     0.5183
★